# Human vs Machine Translation Detection Notebook

## Overview
This notebook implements an automated system to detect whether English translations are **Human Translated (HT)** or **Machine Translated (MT)** using a Large Language Model (LLM) API.

## Key Features
- **LLM-Based Classification**: Uses an OpenAI-compatible API to classify translations
- **Majority Voting**: Runs multiple detection calls per translation and uses majority voting for robust predictions
- **Confidence Scoring**: Provides normalized confidence scores for both HT and MT classifications
- **Batch Processing**: Processes entire CSV files, analyzing selected columns of translations
- **Error Handling**: Implements retry logic with exponential backoff for API failures

## Workflow

### 1. Configuration & Setup
- Initialize OpenAI-compatible client with API credentials
- Set up model parameters and input/output paths

### 2. Core Functions
- **`create_detection_prompt()`**: Generates the prompt for the LLM to classify translations
- **`parse_llm_json()`**: Safely parses JSON responses from the LLM
- **`clean_prediction_dict()`**: Normalizes and validates prediction data
- **`call_detection_api()`**: Single API call with retry mechanism
- **`majority_vote_with_avg_confidence()`**: Multiple calls with voting and confidence averaging

### 3. Data Processing
- **`read_csv_column()`**: Reads a specific column from input CSV
- **`run_detection_on_csv_column()`**: Batch processes all rows in a CSV column and saves results

### 4. Execution & Results
- Process multiple translation columns from input CSV
- Generate output CSVs with predictions and confidence scores
- Display results preview

## Usage
Set `INPUT_CSV` and `MODEL_NAME` variables, then run the batch processing cell to analyze your translations.

In [ ]:
import openai
import pandas as pd
import csv
import time
import json
import re
import os
from typing import Dict, Tuple, List

# Configuration
API_KEY = os.getenv("API_KEY") # Make sure to set this environment variable before running the notebook
BASE_URL = "" # Set this to your OpenAI-compatible API base URL if needed

if not API_KEY:
    raise ValueError("Please set the environment variable API_KEY first.")

# Initialize OpenAI-compatible client
client = openai.OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

In [ ]:
def create_detection_prompt(translated_text: str) -> str:
    """
    Ask the LLM to decide whether an English translation is human translated (HT)
    or machine translated (MT), and return calibrated confidence scores.
    """
    prompt = f"""
You are an expert in translation quality analysis and machine translation detection.

Your task is to examine the following English translated text and decide whether it is:
- HT = Human Translated
- MT = Machine Translated

Return your answer as JSON only, with exactly these keys:
{{
  "label": "HT" or "MT",
  "confidence_ht": float,
  "confidence_mt": float,
}}

Rules:
- confidence_ht and confidence_mt must both be numbers between 0 and 1
- confidence_ht + confidence_mt should equal 1
- Output JSON only
- Do not include markdown
- Do not include extra text

Translated text:
\"\"\"{translated_text}\"\"\"
"""
    return prompt.strip()

In [ ]:
def parse_llm_json(response_text: str) -> Dict:
    """
    Parse JSON response from the LLM safely.
    """
    response_text = response_text.strip()

    # First try direct JSON parse
    try:
        data = json.loads(response_text)
        return data
    except json.JSONDecodeError:
        pass

    # Fallback: extract first JSON object from text
    match = re.search(r"\{.*\}", response_text, flags=re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            return data
        except json.JSONDecodeError:
            pass

    raise ValueError(f"Could not parse JSON from response: {response_text}")


def clean_prediction_dict(data: Dict) -> Dict:
    """
    Normalize label and confidence fields.
    """
    label = str(data.get("label", "")).strip().upper()
    if label not in {"HT", "MT"}:
        raise ValueError(f"Invalid label: {label}")

    conf_ht = float(data.get("confidence_ht", 0.0))
    conf_mt = float(data.get("confidence_mt", 0.0))

    # clamp to [0, 1]
    conf_ht = max(0.0, min(1.0, conf_ht))
    conf_mt = max(0.0, min(1.0, conf_mt))

    total = conf_ht + conf_mt
    if total == 0:
        # fallback to label-based default if the model returns bad scores
        if label == "HT":
            conf_ht, conf_mt = 1.0, 0.0
        else:
            conf_ht, conf_mt = 0.0, 1.0
    else:
        conf_ht /= total
        conf_mt /= total

    return {
        "label": label,
        "confidence_ht": conf_ht,
        "confidence_mt": conf_mt
    }

In [ ]:
def call_detection_api(
    client: openai.OpenAI,
    translated_text: str,
    model_name: str,
    max_retries: int = 3,
    temperature: float = 0.0
) -> Dict:
    """
    One LLM call for HT vs MT classification.
    Returns a dict with label, confidence_ht, confidence_mt, reason.
    """
    prompt = create_detection_prompt(translated_text)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature
            )

            response_text = response.choices[0].message.content.strip()
            parsed = parse_llm_json(response_text)
            cleaned = clean_prediction_dict(parsed)
            cleaned["raw_response"] = response_text
            return cleaned

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Error on attempt {attempt + 1}: {e}. Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                return {
                    "label": "ERROR",
                    "confidence_ht": None,
                    "confidence_mt": None,
                    "reason": f"API Error: {str(e)}",
                    "raw_response": ""
                }

    return {
        "label": "ERROR",
        "confidence_ht": None,
        "confidence_mt": None,
        "reason": "Max retries exceeded",
        "raw_response": ""
    }

In [ ]:
def majority_vote_with_avg_confidence(
    client: openai.OpenAI,
    translated_text: str,
    model_name: str,
    n_calls: int = 3,
    max_retries: int = 3,
    temperature: float = 0.0
) -> Dict:
    """
    Call the model n times, take majority vote for the label,
    and average the confidence scores, then normalize them so they sum to 1.
    """
    results = []

    for _ in range(n_calls):
        result = call_detection_api(
            client=client,
            translated_text=translated_text,
            model_name=model_name,
            max_retries=max_retries,
            temperature=temperature
        )
        results.append(result)

    valid_results = [r for r in results if r["label"] in {"HT", "MT"}]

    if not valid_results:
        return {
            "final_label": "ERROR",
            "final_confidence_ht": None,
            "final_confidence_mt": None
        }

    vote_ht = sum(1 for r in valid_results if r["label"] == "HT")
    vote_mt = sum(1 for r in valid_results if r["label"] == "MT")

    if vote_mt > vote_ht:
        final_label = "MT"
    elif vote_ht > vote_mt:
        final_label = "HT"
    else:
        # tie-break by average confidence
        avg_ht_tmp = sum(r["confidence_ht"] for r in valid_results) / len(valid_results)
        avg_mt_tmp = sum(r["confidence_mt"] for r in valid_results) / len(valid_results)
        final_label = "MT" if avg_mt_tmp > avg_ht_tmp else "HT"

    avg_conf_ht = sum(r["confidence_ht"] for r in valid_results) / len(valid_results)
    avg_conf_mt = sum(r["confidence_mt"] for r in valid_results) / len(valid_results)

    total = avg_conf_ht + avg_conf_mt
    if total == 0:
        avg_conf_ht, avg_conf_mt = (1.0, 0.0) if final_label == "HT" else (0.0, 1.0)
    else:
        avg_conf_ht /= total
        avg_conf_mt /= total

    return {
        "final_label": final_label,
        "final_confidence_ht": avg_conf_ht,
        "final_confidence_mt": avg_conf_mt
    }

In [ ]:
def read_csv_column(source_file: str, col_name: str) -> List[str]:
    """
    Read a specific column from a CSV as a list of strings.
    """
    try:
        sentences = []

        with open(source_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)

            if not reader.fieldnames or col_name not in reader.fieldnames:
                raise ValueError(f"Column '{col_name}' not found in CSV.")

            for row in reader:
                sentence = row.get(col_name, "")
                sentence = "" if sentence is None else str(sentence).strip()
                sentences.append(sentence)

        print(f"Read {len(sentences)} rows from column '{col_name}'.")
        print("-" * 30)
        return sentences

    except FileNotFoundError:
        raise FileNotFoundError("Error: file not found.")

In [ ]:
def run_detection_on_csv_column(
    input_csv_path: str,
    text_column: str,
    output_csv_path: str,
    model_name: str,
    n_calls: int = 3,
    max_retries: int = 3,
    temperature: float = 0.0
) -> pd.DataFrame:
    """
    For each row in the selected text column:
    - run HT/MT detection 3 times
    - do majority voting
    - append results as new columns
    - save to a new CSV
    """
    df = pd.read_csv(input_csv_path)

    if text_column not in df.columns:
        raise ValueError(f"Column '{text_column}' not found in CSV.")

    results = []

    for idx, text in enumerate(df[text_column].fillna("").astype(str).tolist()):
        print(f"Processing row {idx + 1}/{len(df)}")

        pred = majority_vote_with_avg_confidence(
            client=client,
            translated_text=text,
            model_name=model_name,
            n_calls=n_calls,
            max_retries=max_retries,
            temperature=temperature
        )

        results.append(pred)

        time.sleep(0.2)

    pred_df = pd.DataFrame(results)

    safe_model_name = re.sub(r"[^a-zA-Z0-9_]+", "_", model_name.lower())
    pred_df = pred_df.rename(columns={
        col: f"{safe_model_name}_{col}" for col in pred_df.columns
    })

    output_df = pd.concat([df, pred_df], axis=1)
    output_df.to_csv(output_csv_path, index=False, encoding="utf-8")

    print(f"Saved output to: {output_csv_path}")
    return output_df

In [ ]:

INPUT_CSV = "" # Set this to your input CSV file path, which should have the translations you want to analyze.


MODEL_NAME = "gpt-5" # Change this to the model you want to use for detection. "claude-opus-4-20250514-v1:0" or "gemini-2.5-pro"
columns=["human translation","claude","gemini","gemma","llamma","nllb","openai","qwen"]

for col in columns:
    TEXT_COLUMN = col   # change this to any column you want
    OUTPUT_CSV = f"model_{MODEL_NAME[:6]}_col_{TEXT_COLUMN}.csv"

    df_out = run_detection_on_csv_column(
        input_csv_path=INPUT_CSV,
        text_column=TEXT_COLUMN,
        output_csv_path=OUTPUT_CSV,
        model_name=MODEL_NAME,
        n_calls=3,
        max_retries=3,
        temperature=0.0
)


In [ ]:
df_out.head()